# 🚀 Panacee GNN - Kaggle Training Pipeline
## Setup + Phase 1/2/3 Training avec GPU P100

**Ce notebook configure complètement l'environnement et lance l'entraînement.**

---
### ⚙️ Prérequis Kaggle:
1. **Settings** → **Accelerator** = **GPU** (P100)
2. C'est tout ! ✅

---

## ÉTAPE 0️⃣ - Clone du Repo GitHub

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Définir le dossier de travail
WORK_DIR = "/kaggle/working"
os.chdir(WORK_DIR)

# Cloner le repo
print("📥 Clonage du repo GitHub...")
!git clone https://github.com/jumoras0000/savh_gnn.git

PROJECT_DIR = Path(WORK_DIR) / "savh_gnn" / "Projet_Panacee"
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

print(f"✅ Repo cloné dans: {PROJECT_DIR}")
print(f"📁 Fichiers disponibles: {list(PROJECT_DIR.glob('*.py'))[:5]}...")

## ÉTAPE 1️⃣ - Installation des Dépendances (Optimisée Kaggle)

In [ ]:
print("\n" + "="*80)
print("🔧 INSTALLATION DES DÉPENDANCES")
print("="*80)

# 1️⃣ PyTorch (déjà installé sur Kaggle avec CUDA)
print("\n✅ PyTorch déjà pré-installé sur Kaggle avec GPU support")
import torch
print(f"   Version: {torch.__version__}")
print(f"   GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.mem_get_info()[1] / 1e9:.1f} GB")

In [ ]:
# 2️⃣ PyTorch Geometric (important pour ton GNN)
print("\n📦 Installation de PyTorch Geometric...")
!pip install -q torch-geometric

In [ ]:
# 3️⃣ RDKit (chimie moléculaire)
# NB: on installe 'rdkit' (paquet officiel). 'rdkit-pypi' est déprécié et casse souvent.
print("\n📦 Installation de RDKit...")
!pip install -q rdkit

In [ ]:
# 4️⃣ DeepChem (données Tox21 pour Phase 2)
print("\n📦 Installation de DeepChem...")
!pip install -q deepchem

In [ ]:
# 5️⃣ Autres dépendances essentielles
print("\n📦 Installation des autres dépendances...")
!pip install -q scikit-learn pandas numpy scipy matplotlib tqdm beautifulsoup4 lxml biopython python-dotenv psutil packaging

In [ ]:
# Vérification finale
print("\n" + "="*80)
print("✅ VÉRIFICATION DES DÉPENDANCES")
print("="*80)

imports_check = {
    "torch": "PyTorch",
    "torch_geometric": "PyTorch Geometric",
    "rdkit": "RDKit",
    "deepchem": "DeepChem",
    "sklearn": "Scikit-learn",
    "pandas": "Pandas",
    "numpy": "NumPy"
}

for module, name in imports_check.items():
    try:
        __import__(module)
        print(f"✅ {name}")
    except ImportError:
        print(f"❌ {name} - Installation échouée")

print("\n" + "="*80)

## ÉTAPE 2️⃣ - Setup du Projet

In [ ]:
# Créer les dossiers nécessaires
from pathlib import Path

dirs_to_create = [
    PROJECT_DIR / "data" / "raw",
    PROJECT_DIR / "data" / "processed",
    PROJECT_DIR / "data" / "external" / "phase3",
    PROJECT_DIR / "checkpoints" / "phase1",
    PROJECT_DIR / "checkpoints" / "phase2",
    PROJECT_DIR / "checkpoints" / "phase3",
    PROJECT_DIR / "logs",
    PROJECT_DIR / "results"
]

for dir_path in dirs_to_create:
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"📁 {dir_path.relative_to(PROJECT_DIR)}")

print(f"\n✅ Structure du projet créée")

## ÉTAPE 3️⃣ - Sanity Check (Vérifier que tout fonctionne)

In [ ]:
# Vérifier que le code peut être importé (vrais noms de classes/fonctions)
print("\n" + "="*80)
print("🧪 SANITY CHECK")
print("="*80)

ok = True

try:
    from src.config import CHECKPOINT_DIR, EXTERNAL_DIR, PHASE1, PHASE2, PHASE3
    print("✅ Config chargée")
    print(f"   - Checkpoint dir: {CHECKPOINT_DIR}")
    print(f"   - Phase 1 epochs: {PHASE1['epochs']} | Phase 2 epochs: {PHASE2['epochs']}")
except Exception as e:
    ok = False
    print(f"❌ Erreur config: {e}")

try:
    # graph_builder expose des FONCTIONS, pas une classe
    from src.preprocessing.graph_builder import smiles_to_graph, mask_atoms
    g = smiles_to_graph("CC(=O)O")  # test rapide : acide acétique
    assert g is not None and g.x.shape[1] == 9
    print(f"✅ graph_builder OK (acide acétique -> {g.x.shape[0]} atomes, {g.x.shape[1]} features)")
except Exception as e:
    ok = False
    print(f"❌ Erreur graph_builder: {e}")

try:
    from src.models.encoder import MolecularEncoder
    enc = MolecularEncoder()
    n_params = sum(p.numel() for p in enc.parameters())
    print(f"✅ MolecularEncoder OK ({n_params:,} params)")
except Exception as e:
    ok = False
    print(f"❌ Erreur encoder: {e}")

try:
    from src.models.toxicity_classifier import ToxicityClassifier
    from src.preprocessing.toxicity_loader import ToxicityDataset
    print("✅ ToxicityClassifier + ToxicityDataset importés")
except Exception as e:
    ok = False
    print(f"❌ Erreur Phase 2: {e}")

print("\n" + ("✅ Tous les modules critiques sont opérationnels" if ok
              else "⚠️  Au moins un module a échoué — voir erreurs ci-dessus"))

## ÉTAPE 4️⃣ - Configuration du GPU et Memory

In [ ]:
print("\n" + "="*80)
print("🎮 CONFIGURATION GPU")
print("="*80)

import torch

# Configuration GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n📱 Device: {device}")

if torch.cuda.is_available():
    print(f"\n🎮 GPU Info:")
    print(f"   - Nom: {torch.cuda.get_device_name(0)}")
    print(f"   - VRAM totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   - VRAM utilisée: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")
    
    # Optimisations Kaggle
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    print(f"\n✅ Optimisations CUDA activées")
else:
    print("⚠️  GPU non disponible - entraînement sur CPU (LENT)")

print("\n" + "="*80)

## ÉTAPE 5️⃣ - Lancer PHASE 1 (Pré-entraînement GNN optionnel)

**ℹ️ Cette phase crée un encodeur GNN pré-entraîné sur ZINC molecules**

**Durée estimée:** ~1-2h avec P100 GPU  
**Conseil:** Skip si tu as déjà un checkpoint Phase 1

In [ ]:
# Options pour Phase 1 (pré-entraînement MGM sur ZINC)
SKIP_PHASE1 = True       # ⬅️ False pour pré-entraîner (la Phase 2 marche très bien sans)
P1_EPOCHS = 15           # 15 epochs = bon compromis sur Kaggle (config=100 pour run complet)
P1_MAX_MOLECULES = 30000 # sous-ensemble ZINC pour tenir dans le temps Kaggle (None = tout)

if not SKIP_PHASE1:
    print("\n" + "="*80)
    print("🔥 PHASE 1 - PRÉ-ENTRAÎNEMENT GNN (ZINC, MGM)")
    print("="*80)

    import subprocess, sys
    cmd = [sys.executable, "run_phase1.py", "--download",
           "--epochs", str(P1_EPOCHS), "--save_dir", "checkpoints/phase1"]
    if P1_MAX_MOLECULES:
        cmd += ["--max_molecules", str(P1_MAX_MOLECULES)]

    result = subprocess.run(cmd, cwd=str(PROJECT_DIR))
    print("\n✅ Phase 1 terminée" if result.returncode == 0
          else f"\n❌ Phase 1 échouée (exit {result.returncode})")
else:
    print("⏭️  Phase 1 skippée (SKIP_PHASE1=True)")
    print("   → La Phase 2 s'entraîne directement (encodeur from scratch). C'est OK ✅")

## ÉTAPE 6️⃣ - Lancer PHASE 2 (Fine-tuning Toxicité)

**🎯 Objectif:** Fine-tune sur données toxicité (Tox21)  
**Durée estimée:** ~30-45min avec P100 GPU  
**Données:** Auto-téléchargées depuis DeepChem

In [ ]:
print("\n" + "="*80)
print("🔥 PHASE 2 - FINE-TUNING TOXICITÉ (Tox21)")
print("="*80)

import subprocess
import sys

# ── Réglages (modifiables) ───────────────────────────────────────────
P2_EPOCHS = PHASE2['epochs']        # 80 par défaut (early stopping intégré)
P2_BATCH_SIZE = PHASE2['batch_size']
P2_MAX_MOLECULES = None             # ex: 2000 pour un run de test rapide (None = tout Tox21)

cmd = [
    sys.executable, "run_phase2.py",
    "--download",                    # auto-download Tox21 depuis DeepChem
    "--epochs", str(P2_EPOCHS),
    "--batch_size", str(P2_BATCH_SIZE),
    "--patience", str(PHASE2['patience']),
    "--save_dir", "checkpoints/phase2",
]
if P2_MAX_MOLECULES:
    cmd += ["--max_molecules", str(P2_MAX_MOLECULES)]

# NB: la Phase 2 tourne SANS checkpoint Phase 1 (encodeur from scratch).
#     Le meilleur modèle est sauvé dans checkpoints/phase2/best_toxicity_model.pth
result = subprocess.run(cmd, cwd=str(PROJECT_DIR))

if result.returncode == 0:
    print("\n✅ Phase 2 terminée — checkpoints/phase2/best_toxicity_model.pth")
else:
    print(f"\n❌ Phase 2 échouée (exit code: {result.returncode})")

## ÉTAPE 7️⃣ - Lancer PHASE 3 (Multi-propriétés optionnel)

**🎯 Objectif:** Prédire plusieurs propriétés moléculaires simultaneously  
**Durée estimée:** ~20-30min avec P100 GPU  
**Données:** Requires external data or synthetic

In [ ]:
# Options pour Phase 3 (multi-propriétés + IA raisonnement)
SKIP_PHASE3 = True   # ⬅️ False pour lancer Phase 3 (nécessite que la Phase 2 ait fini)

if not SKIP_PHASE3:
    print("\n" + "="*80)
    print("🔥 PHASE 3 - MULTI-PROPRIÉTÉS")
    print("="*80)

    import subprocess, sys
    cmd = [
        sys.executable, "run_phase3.py",
        "--download",                 # télécharge ESOL/Lipo/BBBP/ClinTox/HIV + Tox21
        "--epochs", str(PHASE3['epochs']),
        "--batch_size", str(PHASE3['batch_size']),
        "--save_dir", "checkpoints/phase3",
    ]
    result = subprocess.run(cmd, cwd=str(PROJECT_DIR))
    print("\n✅ Phase 3 terminée" if result.returncode == 0
          else f"\n❌ Phase 3 échouée (exit {result.returncode})")
else:
    print("⏭️  Phase 3 skippée (SKIP_PHASE3=True)")

## ÉTAPE 8️⃣ - Résultats et Checkpoints

In [ ]:
import os
from pathlib import Path

print("\n" + "="*80)
print("📊 RÉSULTATS")
print("="*80)

checkpoints_dir = PROJECT_DIR / "checkpoints"
results_dir = PROJECT_DIR / "results"

print("\n🏆 Checkpoints créés:")
if checkpoints_dir.exists():
    for phase_dir in sorted(checkpoints_dir.glob('*')):
        if phase_dir.is_dir():
            files = list(phase_dir.glob('*.pth'))
            print(f"  📁 {phase_dir.name}/: {len(files)} checkpoints")
            for f in files[:3]:
                size_mb = f.stat().st_size / 1e6
                print(f"     - {f.name} ({size_mb:.1f} MB)")

print("\n📈 Résultats:")
if results_dir.exists():
    files = list(results_dir.glob('*'))
    if files:
        for f in files[:5]:
            print(f"  📄 {f.name}")
    else:
        print("  (aucun résultat encore)")
else:
    print("  (dossier vide)")

print("\n📥 Télécharger depuis Kaggle:")
print(f"  - Checkpoints: /kaggle/working/savh_gnn/Projet_Panacee/checkpoints/")
print(f"  - Résultats: /kaggle/working/savh_gnn/Projet_Panacee/results/")
print(f"  - Logs: /kaggle/working/savh_gnn/Projet_Panacee/logs/")

print("\n" + "="*80)
print("✅ ENTRAÎNEMENT TERMINÉ!")
print("="*80)

## 📝 Notes Finales

### ✅ Ce que ce notebook fait:
1. ✅ Clone ton repo GitHub  
2. ✅ Installe toutes les dépendances (PyTorch, RDKit, DeepChem, etc.)  
3. ✅ Configure le GPU P100 de Kaggle  
4. ✅ Lance Phase 1 (optionnel - pré-entraînement)  
5. ✅ Lance Phase 2 (toxicité) avec données auto-téléchargées  
6. ✅ Lance Phase 3 (optionnel - multi-propriétés)  
7. ✅ Sauvegarde tous les checkpoints et résultats  

### 🎯 Pour utiliser:
1. **Crée un Kaggle Notebook**  
2. **Active le GPU:** Settings → Accelerator = GPU  
3. **Copie ce code** dans le notebook  
4. **Run all cells** (Ctrl+Alt+Enter)  
5. **Attends ~1-2h** pour Phase 2 sur P100 ✅  
6. **Télécharge les checkpoints** depuis `Notebook Output`  

### 💾 Données:
- **Tox21**: Auto-téléchargées depuis DeepChem ✅  
- **ZINC (Phase 1)**: Auto-téléchargées ✅  
- **Aucun upload manuel nécessaire** ✅  

### ⚠️ Limitations Kaggle:
- Max 12h par session (mais Phase 2 prend ~45min)  
- GPU P100 = 16GB VRAM (suffisant pour ce projet)  
- Arrête automatiquement après 12h d'inactivité  

### 🚀 Next Steps:
1. Entraîne sur Kaggle  
2. Télécharge les checkpoints  
3. Utilise-les localement sur VSCode  
4. Push les résultats sur GitHub  

---

**Questions?** Check ton terminal pour les erreurs détaillées! 🔍